In [0]:
%pip install xgboost scikit-learn matplotlib seaborn

In [0]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.preprocessing import LabelEncoder
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import date
import json
from mlflow.models.signature import infer_signature
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType
)

In [0]:
# ── Single widget — all params come in as one JSON string ─────────────────
dbutils.widgets.text("job_parameters", "{}")

parameters = json.loads(dbutils.widgets.get("job_parameters"))

# ── Identity & routing ─────────────────────────────────────────────────────
catalog         = parameters.get("catalog")
source_view     = catalog + "." + parameters.get("source_view")
target_table    = catalog + "." + parameters.get("target_table")
primary_keys    = parameters.get("primary_keys")           # list e.g. ["customer_unique_id"]
job_id          = parameters.get("job_id")
parent_job      = parameters.get("parent_job_id", job_id)
partition       = parameters.get("partition")
default_value_flag = parameters.get("default_value_flag", True)

# ── ML-specific params ─────────────────────────────────────────────────────
feature_cols    = parameters.get("feature_cols")           # list of column names
target_col      = parameters.get("target_col")             # e.g. "churn_label"
model_type      = parameters.get("model_type", "xgboost")  # xgboost | ridge | kmeans
experiment_path = parameters.get("experiment_path")        # MLflow experiment path
test_size       = float(parameters.get("test_size", 0.2))
random_state    = int(parameters.get("random_state", 42))
categorical_cols = parameters.get("categorical_cols", [])  # cols needing encoding e.g. ["customer_state"]
fill_with_zero = parameters.get("fill_with_zero", [])  # whether to fill missing values with

# ── XGBoost hyperparams (with sensible defaults) ───────────────────────────
model_params    = parameters.get("model_params", {
    "n_estimators"     : 300,
    "max_depth"        : 6,
    "learning_rate"    : 0.05,
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "min_child_weight" : 5,
    "gamma"            : 0.1,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 1.5,
    "eval_metric"      : "logloss",
    "random_state"     : random_state,
    "use_label_encoder": False,
})

# ── Threshold tuning ───────────────────────────────────────────────────────
tune_threshold  = parameters.get("tune_threshold", True)   # whether to run threshold tuning
score_col       = parameters.get("score_col", "prediction_probability")  # output score col name
label_col       = parameters.get("label_col", "prediction_label")        # output label col name

print("=" * 55)
print("  PARAMETERS LOADED")
print("=" * 55)
print(f"  Source view    : {source_view}")
print(f"  Target table   : {target_table}")
print(f"  Primary keys   : {primary_keys}")
print(f"  Target column  : {target_col}")
print(f"  Feature count  : {len(feature_cols)}")
print(f"  Model type     : {model_type}")
print(f"  Experiment     : {experiment_path}")
print(f"  Test size      : {test_size}")
print(f"  Job ID         : {job_id}")
print(f"  Partition      : {partition}")
print("=" * 55)

In [0]:
mlflow.set_experiment(experiment_path)
print(f"✅ MLflow experiment : {experiment_path}")

In [0]:
# ── Load source view ───────────────────────────────────────────────────────
df = spark.table(source_view).toPandas()
print(f"✅ Loaded {source_view}")
print(f"   Rows     : {df.shape[0]:,}")
print(f"   Columns  : {df.shape[1]}")

# ── Validate all expected columns exist ───────────────────────────────────
all_expected = feature_cols + [target_col] + primary_keys
missing_cols = [c for c in all_expected if c not in df.columns]

if missing_cols:
    raise ValueError(f"❌ Missing columns in source view: {missing_cols}")

print(f"\n✅ All expected columns present")

# ── Target distribution ────────────────────────────────────────────────────
print(f"\nTarget column   : {target_col}")
print(f"Target distribution:")
print(df[target_col].value_counts(normalize=True).round(4) * 100)

In [0]:
# ── Encode categorical columns ────────────────────────────────────────────
encoders          = {}
encoded_feature_cols = feature_cols.copy()


fill_with_median = [
    col for col in encoded_feature_cols
    if col not in fill_with_zero
]


for col in categorical_cols:
    if col in df.columns:
        le          = LabelEncoder()
        encoded_col = f"{col}_encoded"
        df[encoded_col]  = le.fit_transform(df[col].astype(str))
        encoders[col]    = le
        if col in encoded_feature_cols:
            encoded_feature_cols.remove(col)
        encoded_feature_cols.append(encoded_col)
        print(f"✅ Encoded {col} → {encoded_col} ({df[encoded_col].nunique()} categories)")
    else:
        print(f"⚠️  Skipped {col} — not found in dataframe")

# ── Drop rows where target is null ───────────────────────────────────────
before = len(df)
df     = df[df[target_col].notna()].copy()
print(f"\n✅ Dropped {before - len(df)} rows with null target")
print(f"   Remaining rows : {len(df):,}")

# ── Fill nulls in features with median ───────────────────────────────────
null_counts = df[encoded_feature_cols].isna().sum()
cols_with_nulls = null_counts[null_counts > 0]

if len(cols_with_nulls) > 0:
    print(f"\n⚠️  Null values found in {len(cols_with_nulls)} feature column(s) — filling with median:")
    for col, cnt in cols_with_nulls.items():
        print(f"   {col:<45} {cnt:>6} nulls ({cnt/len(df)*100:.2f}%)")
else:
    print(f"\n✅ No null values found in feature columns")

df[encoded_feature_cols] = df[encoded_feature_cols].fillna(
    df[encoded_feature_cols].median(numeric_only=True)
)

# ── Build X and y ─────────────────────────────────────────────────────────
X = df[encoded_feature_cols]
y = df[target_col].astype(int)

# ── Class distribution ────────────────────────────────────────────────────
n_negative  = (y == 0).sum()
n_positive  = (y == 1).sum()
total       = len(y)
churn_rate  = n_positive / total

print(f"\n{'='*55}")
print(f"  CLASS DISTRIBUTION")
print(f"{'='*55}")
print(f"  Total rows              : {total:,}")
print(f"  Churn = 0 (not churned) : {n_negative:,}  ({n_negative/total*100:.1f}%)")
print(f"  Churn = 1 (churned)     : {n_positive:,}  ({n_positive/total*100:.1f}%)")
print(f"  Churn rate              : {churn_rate:.4f}")
print(f"{'='*55}")

# ── Validate churn rate is within a meaningful range ─────────────────────
if churn_rate > 0.85:
    raise ValueError(
        f"\n❌ CHURN LABEL VALIDATION FAILED\n"
        f"   Churn rate is {churn_rate*100:.1f}% — too high to be meaningful.\n"
        f"   Expected range: 20%–70%\n\n"
        f"   Root cause: churn_label in Gold view is too aggressive.\n"
        f"   Fix: Use the tiered definition —\n"
        f"        multi-order customers  → silent > 90 days\n"
        f"        single-order customers → silent > 180 days"
    )

if churn_rate < 0.05:
    raise ValueError(
        f"\n❌ CHURN LABEL VALIDATION FAILED\n"
        f"   Churn rate is {churn_rate*100:.1f}% — too low to train on.\n"
        f"   Expected range: 20%–70%\n"
        f"   Fix: Relax the churn_label definition in gold.vw_customer_features."
    )

# ── Set scale_pos_weight based on which class is the minority ────────────
if n_positive == 0:
    raise ValueError(
        "❌ No positive samples (churn = 1) found.\n"
        "   Cannot train a churn model — check churn_label definition."
    )

if n_negative == 0:
    raise ValueError(
        "❌ No negative samples (churn = 0) found.\n"
        "   Cannot train a churn model — check churn_label definition."
    )

if n_positive < n_negative:
    # ── Positives are minority → upweight them ────────────────────────────
    scale_pos_weight = round(n_negative / n_positive, 4)
    imbalance_label  = f"positives are minority → upweight by {scale_pos_weight}"

elif n_positive > n_negative:
    # ── Positives are majority → no upweighting needed ───────────────────
    # Setting > 1 would make it worse — cap at 1.0
    scale_pos_weight = 1.0
    imbalance_label  = (
        f"⚠️  positives are MAJORITY ({churn_rate*100:.1f}%) — "
        f"scale_pos_weight set to 1.0\n"
        f"   Recommend fixing churn_label definition in Gold view"
    )

else:
    # ── Perfectly balanced ────────────────────────────────────────────────
    scale_pos_weight = 1.0
    imbalance_label  = "perfectly balanced — scale_pos_weight = 1.0"

model_params["scale_pos_weight"] = scale_pos_weight

print(f"\n  Imbalance direction     : {imbalance_label}")
print(f"  scale_pos_weight        : {scale_pos_weight}")

# ── Final summary ─────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  PREPROCESSING COMPLETE")
print(f"{'='*55}")
print(f"  Feature matrix shape    : {X.shape}")
print(f"  Target shape            : {y.shape}")
print(f"  Categorical cols encoded: {len(encoders)}")
print(f"  scale_pos_weight        : {scale_pos_weight}")
print(f"{'='*55}")# ── Define fill strategies per column ────────────────────────────────────
# Absence-of-activity columns → fill with 0 (not median)
# Missing measurement columns → fill with median

# ── Report nulls before filling ───────────────────────────────────────────
null_counts     = df[encoded_feature_cols].isna().sum()
cols_with_nulls = null_counts[null_counts > 0]

if len(cols_with_nulls) > 0:
    print(f"⚠️  Null values found in {len(cols_with_nulls)} column(s):")
    print(f"{'Column':<45} {'Nulls':>8}  {'%':>6}  {'Fill Strategy'}")
    print("-" * 75)
    for col, cnt in cols_with_nulls.items():
        strategy = "zero  " if col in fill_with_zero else "median"
        print(f"  {col:<43} {cnt:>8}  {cnt/len(df)*100:>5.2f}%  {strategy}")
else:
    print("✅ No null values in feature columns")

# ── Apply fill strategies ─────────────────────────────────────────────────
# Zero fill — absence of activity
zero_cols_present = [c for c in fill_with_zero if c in encoded_feature_cols]
df[zero_cols_present] = df[zero_cols_present].fillna(0)

# Median fill — genuinely missing measurements
median_cols_present = [c for c in fill_with_median if c in encoded_feature_cols]
df[median_cols_present] = df[median_cols_present].fillna(
    df[median_cols_present].median(numeric_only=True)
)

print(f"\n✅ Zero-filled  : {len(zero_cols_present)} columns")
print(f"✅ Median-filled: {len(median_cols_present)} columns")

# ── Confirm no nulls remain ───────────────────────────────────────────────
remaining_nulls = df[encoded_feature_cols].isna().sum().sum()
if remaining_nulls > 0:
    still_null = df[encoded_feature_cols].isna().sum()
    still_null = still_null[still_null > 0]
    raise ValueError(
        f"❌ {remaining_nulls} nulls remain after filling:\n{still_null}"
    )
print(f"✅ Zero nulls remaining in all feature columns")

# ── Build X and y ─────────────────────────────────────────────────────────
# Drop rows where churn_label is NULL
# (recently acquired customers excluded from training)
before_drop = len(df)
df = df[df[target_col].notna()].copy()
dropped     = before_drop - len(df)

if dropped > 0:
    print(f"\n✅ Dropped {dropped:,} rows with null churn_label")
    print(f"   (recently acquired customers — no prediction window)")

X = df[encoded_feature_cols]
y = df[target_col].astype(int)

# ── Class distribution ────────────────────────────────────────────────────
n_negative = (y == 0).sum()
n_positive = (y == 1).sum()
total      = len(y)
churn_rate = n_positive / total

print(f"\n{'='*55}")
print(f"  CLASS DISTRIBUTION")
print(f"{'='*55}")
print(f"  Total rows (after null drop) : {total:,}")
print(f"  Churn = 0 (not churned)      : {n_negative:,}  ({n_negative/total*100:.1f}%)")
print(f"  Churn = 1 (churned)          : {n_positive:,}  ({n_positive/total*100:.1f}%)")
print(f"  Churn rate                   : {churn_rate:.4f}")
print(f"{'='*55}")

# ── Validate churn rate ───────────────────────────────────────────────────
if churn_rate > 0.85:
    raise ValueError(
        f"\n❌ CHURN LABEL VALIDATION FAILED\n"
        f"   Churn rate is {churn_rate*100:.1f}% — still too high.\n"
        f"   Expected range: 20%–70%\n\n"
        f"   Try tightening the observation cutoff date in the Gold view\n"
        f"   e.g. move first_order_date cutoff from 2018-01-01 → 2017-10-01"
    )

if churn_rate < 0.05:
    raise ValueError(
        f"\n❌ CHURN LABEL VALIDATION FAILED\n"
        f"   Churn rate is {churn_rate*100:.1f}% — too low.\n"
        f"   Expected range: 20%–70%\n"
        f"   Try relaxing the churn_label definition in gold.vw_customer_features."
    )

# ── Set scale_pos_weight ──────────────────────────────────────────────────
if n_positive == 0:
    raise ValueError("❌ No positive samples (churn=1). Check churn_label definition.")
if n_negative == 0:
    raise ValueError("❌ No negative samples (churn=0). Check churn_label definition.")

if n_positive < n_negative:
    scale_pos_weight = round(n_negative / n_positive, 4)
    direction        = f"positives are minority → upweight by {scale_pos_weight}"
elif n_positive > n_negative:
    scale_pos_weight = 1.0
    direction        = f"⚠️  positives are MAJORITY ({churn_rate*100:.1f}%) — set to 1.0"
else:
    scale_pos_weight = 1.0
    direction        = "perfectly balanced → set to 1.0"

model_params["scale_pos_weight"] = scale_pos_weight

print(f"\n  Imbalance direction          : {direction}")
print(f"  scale_pos_weight             : {scale_pos_weight}")
print(f"\n{'='*55}")
print(f"  PREPROCESSING COMPLETE")
print(f"{'='*55}")
print(f"  Feature matrix : {X.shape}")
print(f"  Target shape   : {y.shape}")
print(f"  Zero-filled    : {len(zero_cols_present)} cols")
print(f"  Median-filled  : {len(median_cols_present)} cols")
print(f"  scale_pos_weight: {scale_pos_weight}")
print(f"{'='*55}")

In [0]:
# ============================================================
# Column Validation & Type Enforcement
# ============================================================
import pandas as pd
import numpy as np
from decimal import Decimal

# ── Define what XGBoost can and cannot accept ─────────────────────────────
VALID_NUMPY_KINDS  = {"i", "u", "f", "b"}   # int, uint, float, bool
VALID_PANDAS_TYPES = ["int8","int16","int32","int64",
                      "uint8","uint16","uint32","uint64",
                      "float16","float32","float64","bool"]

# ── Conversion priority order ─────────────────────────────────────────────
# For each column we attempt conversions in this order until one succeeds
CONVERSION_PIPELINE = ["float64", "int64", "bool"]

# ── Results tracker ───────────────────────────────────────────────────────
validation_results = []

print("=" * 75)
print(f"  COLUMN VALIDATION REPORT")
print(f"  Source view  : {source_view}")
print(f"  Total feature columns to validate : {len(encoded_feature_cols)}")
print("=" * 75)

# ── Validate & convert each column ───────────────────────────────────────
problem_cols    = []    # cols that failed all conversions
converted_cols  = []    # cols that needed conversion
already_valid   = []    # cols that were already correct dtype

for col in encoded_feature_cols:

    original_dtype = str(df[col].dtype)
    sample_vals    = df[col].dropna().head(3).tolist()
    null_count     = df[col].isna().sum()
    null_pct       = round(null_count / len(df) * 100, 2)
    status         = None
    final_dtype    = original_dtype
    note           = ""

    # ── Step 1: Already a valid numeric type ─────────────────────────────
    if df[col].dtype.kind in VALID_NUMPY_KINDS:
        status     = "✅ VALID"
        final_dtype = original_dtype
        already_valid.append(col)

    # ── Step 2: Object dtype — inspect what's inside ─────────────────────
    elif df[col].dtype == object:

        # Check if it's Decimal objects (from Spark DecimalType)
        non_null = df[col].dropna()
        is_decimal = non_null.apply(lambda x: isinstance(x, Decimal)).any()
        is_numeric_str = False

        if not is_decimal:
            # Check if it's numeric strings e.g. "3.14", "100"
            try:
                pd.to_numeric(non_null.head(100), errors="raise")
                is_numeric_str = True
            except Exception:
                pass

        # Attempt conversion
        converted = False
        for target_type in CONVERSION_PIPELINE:
            try:
                df[col] = pd.to_numeric(df[col], errors="raise").astype(target_type)
                status      = f"🔄 CONVERTED"
                final_dtype = target_type
                note        = f"object ({'Decimal' if is_decimal else 'str'}) → {target_type}"
                converted_cols.append(col)
                converted = True
                break
            except Exception:
                continue

        if not converted:
            status = "❌ FAILED"
            note   = f"object — not Decimal, not numeric string. Sample: {sample_vals[:2]}"
            problem_cols.append(col)

    # ── Step 3: category dtype (pandas categorical) ───────────────────────
    elif str(df[col].dtype) == "category":
        try:
            df[col]     = df[col].cat.codes.astype("int64")
            status      = "🔄 CONVERTED"
            final_dtype = "int64"
            note        = "category → int64 (cat codes)"
            converted_cols.append(col)
        except Exception as e:
            status = "❌ FAILED"
            note   = f"category conversion failed: {str(e)}"
            problem_cols.append(col)

    # ── Step 4: datetime — extract numeric features ───────────────────────
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        try:
            df[col]     = df[col].astype(np.int64) // 10**9   # unix timestamp (seconds)
            status      = "🔄 CONVERTED"
            final_dtype = "int64"
            note        = "datetime64 → unix timestamp (int64)"
            converted_cols.append(col)
        except Exception as e:
            status = "❌ FAILED"
            note   = f"datetime conversion failed: {str(e)}"
            problem_cols.append(col)

    # ── Step 5: boolean-like ──────────────────────────────────────────────
    elif pd.api.types.is_bool_dtype(df[col]):
        df[col]     = df[col].astype("int8")
        status      = "🔄 CONVERTED"
        final_dtype = "int8"
        note        = "bool → int8"
        converted_cols.append(col)

    # ── Step 6: anything else — try brute force to numeric ───────────────
    else:
        try:
            df[col]     = pd.to_numeric(df[col], errors="raise").astype("float64")
            status      = "🔄 CONVERTED"
            final_dtype = "float64"
            note        = f"{original_dtype} → float64"
            converted_cols.append(col)
        except Exception as e:
            status = "❌ FAILED"
            note   = f"{original_dtype} — no valid conversion. Error: {str(e)}"
            problem_cols.append(col)

    # ── Log result ────────────────────────────────────────────────────────
    validation_results.append({
        "column"        : col,
        "original_dtype": original_dtype,
        "final_dtype"   : final_dtype,
        "null_count"    : null_count,
        "null_pct"      : null_pct,
        "status"        : status,
        "note"          : note,
    })

    print(f"  {status:<15} {col:<40} {original_dtype:<12} → {final_dtype:<12} "
          f"nulls={null_pct}%  {note}")

# ── Summary ───────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print(f"  SUMMARY")
print("=" * 75)
print(f"  ✅ Already valid   : {len(already_valid)}")
print(f"  🔄 Converted       : {len(converted_cols)}")
print(f"  ❌ Failed          : {len(problem_cols)}")

if converted_cols:
    print(f"\n  Converted columns  :")
    for c in converted_cols:
        r = next(x for x in validation_results if x["column"] == c)
        print(f"    {c:<40} {r['original_dtype']} → {r['final_dtype']}")

# ── Raise exception if any column could not be converted ─────────────────
if problem_cols:
    problem_detail = "\n".join([
        f"  - {r['column']:<40} dtype={r['original_dtype']}  note={r['note']}"
        for r in validation_results if r["column"] in problem_cols
    ])
    raise TypeError(
        f"\n❌ COLUMN VALIDATION FAILED\n"
        f"The following {len(problem_cols)} column(s) cannot be used for ML training\n"
        f"and could not be automatically converted:\n\n"
        f"{problem_detail}\n\n"
        f"Resolution options:\n"
        f"  1. Remove the column from 'feature_cols' in the job JSON\n"
        f"  2. Add CAST(col AS DOUBLE) in the source Gold view\n"
        f"  3. Add an encoding step before this notebook runs"
    )

# ── Final dtype check — confirm zero object columns remain ───────────────
remaining_objects = df[encoded_feature_cols].select_dtypes(include="object").columns.tolist()
if remaining_objects:
    raise TypeError(
        f"❌ POST-CONVERSION CHECK FAILED\n"
        f"Still object dtype after all conversions: {remaining_objects}"
    )

print(f"\n✅ All {len(encoded_feature_cols)} feature columns are XGBoost-ready")
print(f"   Final dtypes:")
print(df[encoded_feature_cols].dtypes.value_counts().to_string())

# ── Rebuild X and y with clean types ─────────────────────────────────────
X = df[encoded_feature_cols].astype(float)
y = df[target_col].astype(int)

print(f"\n✅ X shape : {X.shape}")
print(f"✅ y shape : {y.shape}")
print(f"✅ y dtype : {y.dtype}")

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = test_size,
    random_state = random_state,
    stratify     = y
)

print(f"Train : {X_train.shape[0]:,} rows | churn rate: {y_train.mean():.4f}")
print(f"Test  : {X_test.shape[0]:,}  rows | churn rate: {y_test.mean():.4f}")

In [0]:
run_name_baseline = f"{model_type}_baseline_{job_id}"

with mlflow.start_run(run_name=run_name_baseline) as run_baseline:

    # ── Tag run with job metadata ──────────────────────────────────────────
    mlflow.set_tags({
        "job_id"       : job_id,
        "parent_job"   : parent_job,
        "partition"    : str(partition),
        "source_view"  : source_view,
        "target_table" : target_table,
        "model_type"   : model_type,
        "run_type"     : "baseline"
    })

    mlflow.log_params(model_params)
    mlflow.log_param("train_size",   X_train.shape[0])
    mlflow.log_param("test_size",    X_test.shape[0])
    mlflow.log_param("n_features",   len(encoded_feature_cols))
    mlflow.log_param("target_col",   target_col)
    mlflow.log_param("source_view",  source_view)

    # ── Train ──────────────────────────────────────────────────────────────
    model_baseline = xgb.XGBClassifier(**model_params)
    model_baseline.fit(
        X_train, y_train,
        eval_set = [(X_test, y_test)],
        verbose  = False
    )

    # ── Evaluate ───────────────────────────────────────────────────────────
    y_proba_b = model_baseline.predict_proba(X_test)[:, 1]
    y_pred_b  = (y_proba_b >= 0.5).astype(int)

    metrics_b = {
        "auc_roc"   : round(roc_auc_score(y_test, y_proba_b), 4),
        "f1_score"  : round(f1_score(y_test, y_pred_b), 4),
        "precision" : round(precision_score(y_test, y_pred_b), 4),
        "recall"    : round(recall_score(y_test, y_pred_b), 4),
    }
    mlflow.log_metrics(metrics_b)

    # ── Feature importance ────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 8))
    xgb.plot_importance(model_baseline, ax=ax, max_num_features=20,
                        title=f"Feature Importance — {run_name_baseline}")
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.close()

    signature     = infer_signature(X_train, model_baseline.predict_proba(X_train)[:, 1])
    input_example = X_train.head(5)

    mlflow.xgboost.log_model(
        model_baseline,
        f"{model_type}_baseline",
        signature     = signature,
        input_example = input_example,
        model_format  = "json"
    )
    baseline_run_id = run_baseline.info.run_id

    print(f"Baseline → AUC: {metrics_b['auc_roc']} | F1: {metrics_b['f1_score']}")

In [0]:
# Override with tighter params for tuned run
tuned_params = {**model_params,
    "n_estimators" : 500,
    "learning_rate": 0.03,
    "max_depth"    : 7,
    "gamma"        : 0.2,
    "reg_alpha"    : 0.2,
    "reg_lambda"   : 2.0,
}

run_name_tuned = f"{model_type}_tuned_{job_id}"

with mlflow.start_run(run_name=run_name_tuned) as run_tuned:

    mlflow.set_tags({
        "job_id"      : job_id,
        "parent_job"  : parent_job,
        "partition"   : str(partition),
        "source_view" : source_view,
        "model_type"  : model_type,
        "run_type"    : "tuned"
    })

    mlflow.log_params(tuned_params)
    mlflow.log_param("train_size",  X_train.shape[0])
    mlflow.log_param("n_features",  len(encoded_feature_cols))

    model_tuned = xgb.XGBClassifier(**tuned_params)
    model_tuned.fit(
        X_train, y_train,
        eval_set = [(X_test, y_test)],
        verbose  = False
    )

    y_proba_t = model_tuned.predict_proba(X_test)[:, 1]
    y_pred_t  = (y_proba_t >= 0.5).astype(int)

    metrics_t = {
        "auc_roc"   : round(roc_auc_score(y_test, y_proba_t), 4),
        "f1_score"  : round(f1_score(y_test, y_pred_t), 4),
        "precision" : round(precision_score(y_test, y_pred_t), 4),
        "recall"    : round(recall_score(y_test, y_pred_t), 4),
    }
    mlflow.log_metrics(metrics_t)

    # ── Threshold tuning (conditional on parameter) ────────────────────────
    best_threshold = 0.5   # default
    if tune_threshold:
        thresholds  = np.arange(0.1, 0.9, 0.05)
        f1_scores   = [
            f1_score(y_test, (y_proba_t >= t).astype(int), zero_division=0)
            for t in thresholds
        ]
        best_threshold = float(thresholds[np.argmax(f1_scores)])
        best_f1        = max(f1_scores)

        mlflow.log_param("best_threshold",    round(best_threshold, 2))
        mlflow.log_metric("best_threshold_f1", round(best_f1, 4))

        fig2, ax2 = plt.subplots(figsize=(10, 5))
        ax2.plot(thresholds, f1_scores, linewidth=2, label="F1")
        ax2.axvline(best_threshold, color="red", linestyle="-.",
                    label=f"Best = {best_threshold:.2f}")
        ax2.set_title(f"Threshold Tuning — {source_view}")
        ax2.legend(); ax2.grid(True, alpha=0.3)
        mlflow.log_figure(fig2, "threshold_tuning.png")
        plt.close()
        print(f"Best threshold : {best_threshold:.2f} | F1: {best_f1:.4f}")

    # ── ROC + Confusion Matrix ────────────────────────────────────────────
    fig3, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 5))
    RocCurveDisplay.from_predictions(
        y_test, y_proba_t, ax=ax3,
        name=f"Tuned (AUC={metrics_t['auc_roc']})"
    )
    ConfusionMatrixDisplay.from_predictions(
        y_test, (y_proba_t >= best_threshold).astype(int),
        ax=ax4, display_labels=["Not Churned", "Churned"], cmap="Blues"
    )
    fig3.suptitle(f"Tuned Model — {source_view}")
    plt.tight_layout()
    mlflow.log_figure(fig3, "roc_confusion_tuned.png")
    plt.close()

    signature     = infer_signature(X_train, model_baseline.predict_proba(X_train)[:, 1])
    input_example = X_train.head(5)

    mlflow.xgboost.log_model(
        model_tuned,
        f"{model_type}_tuned",
        signature     = signature,
        input_example = input_example,
        model_format  = "json")
    tuned_run_id = run_tuned.info.run_id

    print(f"Tuned  → AUC: {metrics_t['auc_roc']} | F1: {metrics_t['f1_score']}")

In [0]:
print(f"Searching experiment : {repr(experiment_path)}")
print(f"Filtering by job_id  : {repr(job_id)}")

In [0]:
# ── Compare baseline vs tuned ──────────────────────────────────────────────
runs_df = mlflow.search_runs(
    experiment_names = [experiment_path],
    filter_string    = f"tags.job_id = '{job_id}'",   # only runs from THIS job
    order_by         = ["metrics.auc_roc DESC"]
)

best_run    = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_auc    = best_run["metrics.auc_roc"]
best_name   = best_run["tags.mlflow.runName"]

print(f"\nRun comparison for job_id={job_id}:")
print(
    runs_df[[
        "tags.mlflow.runName", "metrics.auc_roc",
        "metrics.f1_score",    "metrics.precision",
        "metrics.recall"
    ]].to_string(index=False)
)
print(f"\n✅ Best run  : {best_name}")
print(f"✅ AUC-ROC   : {best_auc:.4f}")

# ── Register best model ───────────────────────────────────────────────────
model_uri     = f"runs:/{best_run_id}/{model_type}_tuned"
model_name    = f"{catalog}_{target_col}_{model_type}_{partition}"

model_details = mlflow.register_model(model_uri, model_name)

print(f"\n✅ Registering model '{model_name}' to model registry completed successfully..")

In [0]:
now = pd.Timestamp.now().to_pydatetime()

# ── Helper to safely extract metrics ─────────────────────────────────────
def safe_metric(col):
    val = best_run.get(f"metrics.{col}", None)
    return float(val) if pd.notna(val) else None

# ── Explicit schema — prevents CANNOT_DETERMINE_TYPE on None values ───────
registry_schema = StructType([
    # Model identity
    StructField("model_name",       StringType(),    nullable=False),
    StructField("model_type",       StringType(),    nullable=True),
    StructField("target_col",       StringType(),    nullable=True),
    StructField("source_view",      StringType(),    nullable=True),

    # MLflow references
    StructField("run_id",           StringType(),    nullable=False),
    StructField("model_uri",        StringType(),    nullable=True),
    StructField("mlflow_version",   StringType(),    nullable=True),

    # Performance
    StructField("auc_roc",          DoubleType(),    nullable=True),
    StructField("f1_score",         DoubleType(),    nullable=True),
    StructField("precision_score",  DoubleType(),    nullable=True),
    StructField("recall_score",     DoubleType(),    nullable=True),
    StructField("cv_auc_mean",      DoubleType(),    nullable=True),
    StructField("cv_auc_std",       DoubleType(),    nullable=True),

    # Inference config
    StructField("best_threshold",   DoubleType(),    nullable=True),
    StructField("score_col",        StringType(),    nullable=True),
    StructField("label_col",        StringType(),    nullable=True),
    StructField("feature_cols",     StringType(),    nullable=True),
    StructField("primary_keys",     StringType(),    nullable=True),

    # Job metadata
    StructField("job_id",           StringType(),    nullable=True),
    StructField("parent_job_id",    StringType(),    nullable=True),
    StructField("partition",        StringType(),    nullable=True),

    # Lifecycle
    StructField("status",           StringType(),    nullable=True),
    StructField("registered_at",    TimestampType(), nullable=True),
    StructField("updated_at",       TimestampType(), nullable=True),
    StructField("retired_at",       TimestampType(), nullable=True),
    StructField("registered_by",    StringType(),    nullable=True),
])

# ── Build row — all None values now typed correctly via schema ────────────
registry_row = [(
    # Model identity
    str(model_name),
    str(model_type),
    str(target_col),
    str(source_view),

    # MLflow references
    str(best_run_id),
    str(model_uri),
    str(model_details.version) if model_details else None,

    # Performance
    safe_metric("auc_roc"),
    safe_metric("f1_score"),
    safe_metric("precision"),
    safe_metric("recall"),
    safe_metric("cv_auc_mean"),
    safe_metric("cv_auc_std"),

    # Inference config
    float(best_threshold) if "best_threshold" in dir() else 0.45,
    str(score_col),
    str(label_col),
    json.dumps(encoded_feature_cols),
    json.dumps(primary_keys),

    # Job metadata
    str(job_id),
    str(parent_job),
    str(partition),

    # Lifecycle
    "production",
    now,
    now,
    None,           # retired_at — None is safe now because schema says TimestampType
    "sahil.prusty09@gmail.com",
)]

# ── Create DataFrame with explicit schema ─────────────────────────────────
registry_df    = spark.createDataFrame(registry_row, schema=registry_schema)
registry_table = f"{catalog}.gold.model_registry"

print("✅ Registry DataFrame created")
print(f"   Schema : {registry_df.dtypes}")

# ── MERGE — upsert by model_name ──────────────────────────────────────────
DeltaTable.forName(spark, registry_table) \
    .alias("t") \
    .merge(
        registry_df.alias("s"),
        "t.model_name = s.model_name"
    ) \
    .whenMatchedUpdate(set={
        "run_id"          : "s.run_id",
        "model_uri"       : "s.model_uri",
        "mlflow_version"  : "s.mlflow_version",
        "auc_roc"         : "s.auc_roc",
        "f1_score"        : "s.f1_score",
        "precision_score" : "s.precision_score",
        "recall_score"    : "s.recall_score",
        "cv_auc_mean"     : "s.cv_auc_mean",
        "cv_auc_std"      : "s.cv_auc_std",
        "best_threshold"  : "s.best_threshold",
        "feature_cols"    : "s.feature_cols",
        "status"          : "s.status",
        "updated_at"      : "s.updated_at",
        "job_id"          : "s.job_id",
        "partition"       : "s.partition",
    }) \
    .whenNotMatchedInsertAll() \
    .execute()

print(f"✅ Model registry updated : {registry_table}")
print(f"\n📋 Current registry:")
display(spark.table(registry_table).filter(f"model_name = '{model_name}'").orderBy("updated_at", ascending=False))